#### Install required dependencies

In [2]:
!pip install pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 3.3 MB/s eta 0:00:00a 0:00:01


In [3]:
!pip install mistralai

  Using cached eval_type_backport-0.2.2-py3-none-any.whl.metadata (2.2 kB)
  Using cached jsonpath_python-1.0.6-py3-none-any.whl.metadata (12 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.27.2-cp310-cp310-macosx_11_0_arm64.whl.metadata (6.6 kB)
Using cached eval_type_backport-0.2.2-py3-none-any.whl (5.8 kB)
Using cached jsonpath_python-1.0.6-py3-none-any.whl (7.6 kB)
Using cached pydantic_core-2.27.2-cp310-cp310-macosx_11_0_arm64.whl (1.8 MB)
Using cached typing_inspect-0.9.0-py3-none-any.whl (8.8 kB)
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached h11-0.14.0-py3-none-any.whl (58 kB)


In [5]:
!pip install requests

  Using cached requests-2.32.3-py3-none-any.whl.metadata (4.6 kB)
Using cached requests-2.32.3-py3-none-any.whl (64 kB)


### Running Experiments and Evaluating `Pixtral-12B` Across Various Methods

In [2]:
from PIL import Image

import base64
import os
import requests

import pandas as pd
import numpy as np
import time

from pydantic import BaseModel

from mistralai import Mistral
from utils import read_red_channel, read_green_channel, read_blue_channel, generate_random_sample

In [4]:
# Helper function
def encode_image(image_path):
    """Encode the image to base64."""
    try:
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')
    except FileNotFoundError:
        print(f"Error: The file {image_path} was not found.")
        return None
    except Exception as e:  # Added general exception handling
        print(f"Error: {e}")
        return None

# Specify model
model = "pixtral-12b-2409"

# Path to your image
image_path = "data/converted/19.jpg"

# Getting the base64 string
base64_image = encode_image(image_path)

api_key = os.environ["PIXTRAL"]
client = Mistral(api_key=api_key)

## Demo

In [5]:
# Demo
demo_img = '/Users/mohit/Documents/GitHub/ecdna-analysis/test_im_data/labels/1214.png'
# Encode image
img_input = encode_image(demo_img)

# Define the messages for the chat
messages = [
    {
        "role":"system",
        "content": "You are a pathologist analyzing metaphase cell images. These images contain three main structures: ecDNA, nuclei, and chromosomes. Your goal is to focus on connected component analysis in the image."
        
    },
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "How many ecDNA are in the image? Return an integer number."
            },
            {
                "type": "image_url",
                "image_url": f"data:image/jpeg;base64,{img_input}" 
            }
        ]
    }
]

# Get the chat response
chat_response = client.chat.complete(
    model=model,
    temperature=0.7,
    messages=messages
)

# Print the content of the response
print(chat_response.choices[0].message.content)

The image shows multiple structures, but it is not clear what specific visual characteristics define "ecDNA" in this context. Assuming "ecDNA" refers to the green structures in the image, there are 10 distinct green structures visible.


### Generate and load data

In [6]:
# Helper functions to generate metrics

def filter_data(actual, predicted, threshold = 1000):
    mask_0 = actual != 0
    actual = actual[mask_0]
    predicted = predicted[mask_0]

    threshold_mask = predicted < threshold
    actual = actual[threshold_mask]
    predicted = predicted[threshold_mask]
    
    return actual, predicted
    
def calc_metrics(actual, predicted):
    actual, predicted = filter_data(actual, predicted)

    RMSE = np.sqrt(((predicted - actual) ** 2).mean())
    MAE = (((predicted - actual)).abs()).mean()
    MAPE = (((predicted - actual) / actual).abs()).mean()
    
    return MAE, RMSE, MAPE

In [7]:
# Load train data into a df
HOME = '/Users/mohit/Documents/GitHub/ecdna-analysis'
train_df = pd.read_csv(HOME + "/train_im_data" + '/train_ec_quantification.csv', header=0, names=['img','ec'])
train_df['img'] = train_df['img'].apply(lambda x: x[:-4])
train_df.head()

,img,ec
0,2343,0
1,2344,0
2,2345,0
3,2346,0
4,2347,1


In [8]:
# Load the test data
test_df = pd.read_csv(HOME + "/test_im_data" + '/test_im_ec_quantification.csv', header=0, names=['img','ec'])
test_df['img'] = test_df['img'].apply(lambda x: x[:-4])
test_df.head()

,img,ec
0,0,1
1,1,3
2,10,2
3,1000,2
4,1001,0


In [15]:
def generate_response(client, model, img_path, prompt, context, response_format=None, temp=0.7):
    
    # Encode image
    img_input = encode_image(img_path)
    
    # Define the messages for the chat
    messages = [
        {
            "role":"system",
            "content": f'{context}'
            
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"{prompt}"
                },
                {
                    "type": "image_url",
                    "image_url": f"data:image/jpeg;base64,{img_input}" 
                }
            ]
        }
    ]
    
    # Get the chat response
    chat_response = client.chat.complete(
        model=model,
        temperature=temp,
        messages=messages
    )

    # Return the content of the response
    return chat_response.choices[0].message.content

### Experiment #1: 0-Shot Learning

In [16]:
def pixtral_zero_shot(data_folder):
    ec_predictions = []
    context = """
            For this task, you will act as a pathologist who is studying 
            extrachromosomal DNA (ecDNA). You will be given multiple images
            and asked to identify the number of circular ecDNA structures.
            ecDNA is usually smaller than chromosomes or nuclei which are also present.
            """
    prompt = "Count the number of ecDNA in this image. Output only a single integer and no additional text."
    
    model = "pixtral-12b-2409"
    api_key = os.environ["PIXTRAL"]
    client = Mistral(api_key=api_key)
    img_list = os.listdir(data_folder)
    
    for img in img_list:
        if img.endswith('.png'):    
            print(img)
            
            img_path = os.path.join(data_folder, img)
            result = generate_response(client, model, img_path, prompt, context)
            ec_predictions.append(result)
            
            time.sleep(3)
        
    preds = pd.DataFrame(data={'img':img_list, 'pred':ec_predictions})
    preds['img'] = preds['img'].apply(lambda x: x[:-4])
    return preds

In [17]:
test_folder = '/Users/mohit/Documents/GitHub/ecdna-analysis/test_im_data/labels'
sampled_path = './data/sampled'
sample_df = generate_random_sample(input_folder=test_folder, output_folder=sampled_path, sample_size=100)

preds = pixtral_zero_shot(sampled_path)


1637.png
770.png
228.png
765.png
1150.png
2317.png
388.png
1230.png
2249.png
1754.png
766.png
1988.png
1750.png
549.png
58.png
373.png
615.png
1593.png
2110.png
9.png
1394.png
1169.png
14.png
276.png
470.png
328.png
1694.png
1327.png
704.png
248.png
1910.png
1092.png
1051.png
2149.png
675.png
1450.png
271.png
700.png
892.png
702.png
925.png
266.png
298.png
930.png
478.png
108.png
134.png
914.png
733.png
653.png
447.png
1886.png
490.png
916.png
903.png
23.png
1893.png
245.png
641.png
331.png
469.png
496.png
643.png
858.png
2153.png
247.png
253.png
871.png
369.png
625.png
745.png
235.png
1992.png
744.png
829.png
544.png
633.png
1628.png
343.png
409.png
145.png
44.png
743.png
2047.png
2250.png
226.png
795.png
811.png
45.png
636.png
2324.png
2326.png
84.png
191.png
2285.png
1772.png
1558.png
782.png
806.png
1376.png


In [18]:
baseline_df= pd.merge(preds, test_df, how='left', on='img')
baseline_df.to_csv('./results/zero_shot.csv', index=False)
baseline_df

,img,pred,ec
0,1637,5,10
1,770,2,3
2,228,5,7
3,765,2,0
4,1150,2,3
...,...,...,...
95,1772,1,0
96,1558,3,6
97,782,1,0
98,806,1,3


In [19]:
# Calculate the metrics
calc_metrics(baseline_df['ec'].astype(int), baseline_df['pred'].astype(int))

(np.float64(10.242424242424242),
 np.float64(21.47796615843852),
 np.float64(0.5351791634352739))

### Experiment #2: 3-shot Learning

In [24]:
def generate_three_shot_response(client, model, three_shot_imgs, input_img_path, prompt, context, temp=0.7):
    
    # Encode image
    img_input = encode_image(input_img_path)
    
    # Define the messages for the chat
    messages = [
        {
            "role":"system",
            "content": f'{context}'
            
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Here is the first image which has 1 ecDNA."
                },
                {
                    "type": "image_url",
                    "image_url": f"data:image/jpeg;base64,{three_shot_imgs[0]}" 
                }
            ]
        },
        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": "Understood."
                }
            ]
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Here is the second image which has 19 ecDNA."
                },
                {
                    "type": "image_url",
                    "image_url": f"data:image/jpeg;base64,{three_shot_imgs[1]}" 
                }
            ]
        },
        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": "Understood."
                }
            ]
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Here is the first image which has 129 ecDNA."
                },
                {
                    "type": "image_url",
                    "image_url": f"data:image/jpeg;base64,{three_shot_imgs[2]}" 
                }
            ]
        },
        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": "Got it, I am ready for the final input image."
                }
            ]
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"{prompt}"
                },
                {
                    "type": "image_url",
                    "image_url": f"data:image/jpeg;base64,{img_input}" 
                }
            ]
        }
    ]
    
    # Get the chat response
    chat_response = client.chat.complete(
        model=model,
        temperature=temp,
        messages=messages
    )

    # Return the content of the response
    return chat_response.choices[0].message.content

In [27]:
def pixtral_three_shot(data_folder):
    ec_predictions = []
    context = """
                For this task, you will act as a pathologist who is studying 
                extrachromosomal DN A (ecDNA). You will be given multiple images
                and asked to identify the number of circular ecDNA structures.
                ecDNA is usually smaller than chromosomes or nuclei which are also present.
                I will first provide 3 example images and counts.
            """
    prompt = "Count the number of ecDNA in this image. Output only a single integer and no additional text."
    
    model = "pixtral-12b-2409"
    api_key = os.environ["PIXTRAL"]
    client = Mistral(api_key=api_key)
    img_list = os.listdir(data_folder)
    
    
    # Encode context images
    img_refs = ['2347.png', '2573.png', '4714.png']
    context_imgs = [encode_image(os.path.join('/Users/mohit/Documents/GitHub/ecdna-analysis/train_im_data/labels/', img)) for img in img_refs]
    
    for img in img_list:
        if img.endswith('.png'):    
            print(img)
            
            img_path = os.path.join(data_folder, img)
            result = generate_three_shot_response(client, model, context_imgs, img_path, prompt, context)
            ec_predictions.append(result)
            
            time.sleep(3)
        
    preds = pd.DataFrame(data={'img':img_list, 'pred':ec_predictions})
    preds['img'] = preds['img'].apply(lambda x: x[:-4])
    return preds

In [28]:
test_folder = '/Users/mohit/Documents/GitHub/ecdna-analysis/test_im_data/labels'
sampled_path = './data/sampled'
sample_df = generate_random_sample(input_folder=test_folder, output_folder=sampled_path, sample_size=100)

preds = pixtral_three_shot(sampled_path)

1637.png
837.png
1233.png
1970.png
1031.png
215.png
2049.png
388.png
834.png
767.png
2248.png
969.png
835.png
438.png
1180.png
1793.png
2113.png
70.png
824.png
1426.png
1381.png
1430.png
8.png
2139.png
1550.png
205.png
1394.png
1814.png
1053.png
699.png
672.png
1131.png
704.png
274.png
1051.png
1722.png
1332.png
315.png
1134.png
1444.png
517.png
270.png
13.png
1337.png
890.png
1848.png
662.png
138.png
1731.png
515.png
1297.png
1485.png
1888.png
1846.png
1649.png
652.png
685.png
295.png
2235.png
309.png
725.png
1449.png
1893.png
495.png
1129.png
1074.png
245.png
133.png
641.png
1102.png
131.png
2190.png
2153.png
246.png
736.png
1329.png
369.png
591.png
546.png
977.png
94.png
1159.png
395.png
2042.png
949.png
544.png
545.png
790.png
1837.png
192.png
1994.png
2046.png
795.png
2330.png
620.png
90.png
2130.png
1983.png
1558.png
812.png


In [29]:
three_shot_df= pd.merge(preds, test_df, how='left', on='img')
three_shot_df.to_csv('./results/three_shot.csv', index=False)
three_shot_df

,img,pred,ec
0,1637,1,10
1,837,2,0
2,1233,1,4
3,1970,1,8
4,1031,1,3
...,...,...,...
95,90,1,16
96,2130,1,13
97,1983,1,2
98,1558,1,6


In [30]:
# Calculate the metrics
calc_metrics(three_shot_df['ec'].astype(int), three_shot_df['pred'].astype(int))

(np.float64(10.225352112676056),
 np.float64(17.099542865152365),
 np.float64(0.6236027454563838))

### Experiment #3: Multi-Layer Prompting

In [31]:
def generate_multilayer_response(client, model, input_img_path, prompt, context, temp=0.7):
    
    # Encode image
    img_input = encode_image(input_img_path)
    
    # Define the messages for the chat
    messages = [
        {
            "role":"system",
            "content": f'{context}'
            
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Describe this image."
                },
                {
                    "type": "image_url",
                    "image_url": f"data:image/jpeg;base64,{img_input}" 
                }
            ]
        },
        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": "This is a stained image of cells containing various structures such as ecDNA, chromosomes, and nuclei."
                }
            ]
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"{prompt}"
                }
            ]
        }
    ]
    
    # Get the chat response
    chat_response = client.chat.complete(
        model=model,
        temperature=temp,
        messages=messages
    )

    # Return the content of the response
    return chat_response.choices[0].message.content

In [32]:
def pixtral_multi_layer(data_folder):
    ec_predictions = []
    context = """
            For this task, you will act as a pathologist who is studying 
            extrachromosomal DNA (ecDNA). You will be given multiple images
            and asked to identify the number of circular ecDNA structures.
            ecDNA is usually smaller than chromosomes or nuclei which are also present.
            """
    prompt = "Count the number of ecDNA in this image. Output only a single integer and no additional text."
    
    model = "pixtral-12b-2409"
    api_key = os.environ["PIXTRAL"]
    client = Mistral(api_key=api_key)
    img_list = os.listdir(data_folder)
    
    for img in img_list:
        if img.endswith('.png'):    
            print(img)
            
            img_path = os.path.join(data_folder, img)
            result = generate_multilayer_response(client, model, img_path, prompt, context)
            ec_predictions.append(result)
            
            time.sleep(3.5)
        
    preds = pd.DataFrame(data={'img':img_list, 'pred':ec_predictions})
    preds['img'] = preds['img'].apply(lambda x: x[:-4])
    return preds


In [33]:
test_folder = '/Users/mohit/Documents/GitHub/ecdna-analysis/test_im_data/labels'
sampled_path = './data/sampled'
sample_df = generate_random_sample(input_folder=test_folder, output_folder=sampled_path, sample_size=100)

preds = pixtral_multi_layer(sampled_path)

1810.png
1179.png
1227.png
228.png
1025.png
215.png
1033.png
969.png
49.png
438.png
1590.png
2264.png
2259.png
1793.png
1197.png
826.png
1785.png
1008.png
748.png
934.png
1250.png
1720.png
510.png
276.png
262.png
706.png
658.png
710.png
705.png
1332.png
895.png
329.png
1493.png
12.png
271.png
2213.png
1484.png
1309.png
10.png
313.png
1891.png
691.png
1662.png
725.png
2021.png
902.png
269.png
1070.png
2236.png
1273.png
1313.png
442.png
1129.png
536.png
293.png
245.png
537.png
906.png
2151.png
1302.png
325.png
119.png
694.png
1248.png
1089.png
509.png
290.png
1301.png
130.png
656.png
440.png
1428.png
2069.png
2295.png
94.png
626.png
1588.png
551.png
790.png
974.png
1370.png
54.png
1827.png
804.png
757.png
233.png
569.png
596.png
2250.png
1771.png
1765.png
232.png
393.png
2130.png
1773.png
1001.png
594.png
225.png
0.png
345.png


In [34]:
multi_layer_df= pd.merge(preds, test_df, how='left', on='img')
multi_layer_df.to_csv('./results/multilayer.csv', index=False)
multi_layer_df

,img,pred,ec
0,1810,6,17
1,1179,1,0
2,1227,1,0
3,228,3,7
4,1025,1,1
...,...,...,...
95,1001,1,0
96,594,15,154
97,225,10,24
98,0,1,1


In [35]:
# Calculate the metrics
calc_metrics(multi_layer_df['ec'].astype(int), multi_layer_df['pred'].astype(int))

(np.float64(12.455882352941176),
 np.float64(26.625120822702936),
 np.float64(0.5408254844951226))

### Experiment #4: Temperature Adjustment

In [36]:
def generate_temp_response(client, model, img_path, prompt, context, response_format=None, temp=0.7):
    
    # Encode image
    img_input = encode_image(img_path)
    
    # Define the messages for the chat
    messages = [
        {
            "role":"system",
            "content": f'{context}'
            
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"{prompt}"
                },
                {
                    "type": "image_url",
                    "image_url": f"data:image/jpeg;base64,{img_input}" 
                }
            ]
        }
    ]
    
    # Get the chat response
    chat_response = client.chat.complete(
        model=model,
        temperature=temp,
        messages=messages
    )

    # Return the content of the response
    return chat_response.choices[0].message.content

In [37]:
def pixtral_temp_change(data_folder):
    ec_predictions = []
    context = """
            For this task, you will act as a pathologist who is studying 
            extrachromosomal DNA (ecDNA). You will be given multiple images
            and asked to identify the number of circular ecDNA structures.
            ecDNA is usually smaller than chromosomes or nuclei which are also present.
            """
    prompt = "Count the number of ecDNA in this image. Output only a single integer and no additional text."
    
    model = "pixtral-12b-2409"
    api_key = os.environ["PIXTRAL"]
    client = Mistral(api_key=api_key)
    img_list = os.listdir(data_folder)
    
    for img in img_list:
        if img.endswith('.png'):    
            print(img)
            
            img_path = os.path.join(data_folder, img)
            result = generate_temp_response(client, model, img_path, prompt, context, temp=0.2)
            ec_predictions.append(result)
            
            time.sleep(3.5)
        
    preds = pd.DataFrame(data={'img':img_list, 'pred':ec_predictions})
    preds['img'] = preds['img'].apply(lambda x: x[:-4])
    
    return preds

In [38]:
test_folder = '/Users/mohit/Documents/GitHub/ecdna-analysis/test_im_data/labels'
sampled_path = './data/sampled'
sample_df = generate_random_sample(input_folder=test_folder, output_folder=sampled_path, sample_size=100)

preds = pixtral_temp_change(sampled_path)

1192.png
189.png
566.png
1226.png
149.png
570.png
1190.png
1143.png
399.png
762.png
1553.png
171.png
617.png
2339.png
2313.png
615.png
629.png
72.png
211.png
1169.png
1047.png
1709.png
538.png
699.png
1494.png
672.png
2189.png
1133.png
1694.png
2175.png
1468.png
113.png
715.png
1732.png
2213.png
1069.png
700.png
13.png
648.png
474.png
448.png
689.png
2006.png
273.png
1334.png
449.png
646.png
685.png
929.png
242.png
1933.png
1270.png
1266.png
1059.png
1930.png
1267.png
903.png
687.png
37.png
654.png
873.png
2232.png
1262.png
682.png
643.png
2033.png
1711.png
871.png
1473.png
1428.png
194.png
2134.png
785.png
2297.png
1775.png
747.png
196.png
6.png
394.png
1827.png
1189.png
2325.png
93.png
192.png
1214.png
970.png
555.png
2046.png
191.png
1377.png
754.png
768.png
542.png
2291.png
1001.png
1982.png
2284.png
1969.png
796.png
1570.png


In [39]:
temp_df= pd.merge(preds, test_df, how='left', on='img')
temp_df.to_csv('./results/temperature.csv', index=False)
temp_df

,img,pred,ec
0,1192,3,8
1,189,12,96
2,566,15,42
3,1226,1,2
4,149,45,80
...,...,...,...
95,1982,0,0
96,2284,1,0
97,1969,3,14
98,796,1,0


In [40]:
# Calculate the metrics
calc_metrics(temp_df['ec'].astype(int), temp_df['pred'].astype(int))

(np.float64(15.753846153846155),
 np.float64(30.67196163875455),
 np.float64(0.5814725644666963))